# Recommendation System
Three approaches are compared using existing engineered features: KNN, cosine similarity, and transparent weighted compatibility.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
from src.recommendation import knn_recommend, cosine_recommend, weighted_recommend

cleaned=pd.read_csv(PROJECT_DIR/'data/dormmatch_cleaned.csv')
raw=pd.read_csv(PROJECT_DIR/'data/dormmatch_raw.csv')
scaled=pd.read_csv(PROJECT_DIR/'data/dormmatch_scaled_features.csv')
assignments=pd.read_csv(PROJECT_DIR/'models/cluster_assignments.csv')

raw_profile=raw[['student_id','guest_frequency','music_frequency']].drop_duplicates('student_id',keep='first')
raw_profile['guest_frequency']=raw_profile['guest_frequency'].astype('string').str.strip().str.lower()
raw_profile['music_frequency']=raw_profile['music_frequency'].astype('string').str.strip().str.lower()
candidates=cleaned.merge(raw_profile,on='student_id',how='left',validate='one_to_one').merge(assignments,on='student_id',how='left',validate='one_to_one')
print(candidates.shape)
candidates.head()

(2000, 27)


,student_id,age,gender,year_of_study,branch,cgpa,sleep_time,wake_time,sleep_duration,cleanliness_level,...,smoking_preference,ac_preference,preferred_room_type,budget,preferred_environment,roommate_interaction,calculated_sleep_duration,guest_frequency,music_frequency,cluster
0,STU0760,19,Female,3,Biotechnology,9.03,2.35,6.71,4.07,3.0,...,No smoking,Yes,Double,6500.0,Balanced,2,4.36,often,rarely,0
1,STU1501,21,Male,1,Electrical Engineering,7.92,23.50,6.98,7.31,4.0,...,No smoking,Yes,Triple,6500.0,Balanced,4,7.48,sometimes,rarely,1
2,STU1385,20,Female,4,Computer Science,8.93,22.99,7.18,8.47,5.0,...,No smoking,No,Double,6500.0,Social,2,8.19,sometimes,sometimes,1
3,STU0052,20,Male,2,Computer Science,7.45,22.25,5.81,7.90,4.0,...,No smoking,Yes,Single,5500.0,quiet,4,7.56,often,often,1
4,STU0660,20,Female,2,Electrical Engineering,7.02,0.77,10.68,10.38,2.0,...,No preference,Yes,Double,6000.0,Balanced,3,9.91,sometimes,often,0


## KNN
NearestNeighbors uses Euclidean distance. In the standardized feature matrix, larger values indicate larger lifestyle distance. The query student is excluded.

In [2]:
example_index=0
knn_ids,knn_distances=knn_recommend(scaled.to_numpy(),top_k=5,query_index=example_index)
pd.DataFrame({'student_id':candidates.iloc[knn_ids]['student_id'].values,'distance':knn_distances})

,student_id,distance
0,STU0721,3.010913
1,STU0504,3.440692
2,STU1432,3.494416
3,STU0038,3.583444
4,STU0317,3.623164


## Cosine similarity
Cosine similarity measures vector angle and is independent of vector magnitude: $cos(\theta)=x\cdot y/(||x||||y||)$.

In [3]:
cosine_ids,cosine_scores=cosine_recommend(scaled.to_numpy(),top_k=5,query_index=example_index)
pd.DataFrame({'student_id':candidates.iloc[cosine_ids]['student_id'].values,'cosine_similarity':cosine_scores})

/Users/ananyaa/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/ananyaa/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/ananyaa/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


,student_id,cosine_similarity
0,STU0721,0.841961
1,STU0504,0.797512
2,STU1432,0.776883
3,STU0038,0.773699
4,STU1553,0.747463


## Weighted compatibility
The weighted score combines sleep timing, study, cleanliness, noise, social life, guest habits, music, and preferences. Sleep and wake differences use circular 24-hour distance. The 0-100 score is transparent and CGPA/branch are not inputs.

In [4]:
query=candidates.iloc[example_index]
weighted=weighted_recommend(query,candidates,top_k=5)
weighted

,overall,sleep,study,cleanliness,noise,social,guest,music,preferences,sleep_difference_hours,student_id,cluster
0,91.925000,87.541667,98.333333,100.0,75.0,100.0,100.000000,66.666667,100.0,1.495,STU1710,0
1,91.875000,84.291667,97.888889,100.0,100.0,100.0,66.666667,66.666667,100.0,1.885,STU0093,0
2,91.591667,91.791667,95.444444,75.0,100.0,100.0,100.000000,66.666667,100.0,0.985,STU0317,0
3,91.433333,91.500000,94.222222,100.0,75.0,100.0,100.000000,100.000000,75.0,1.020,STU0076,0
4,90.675000,92.833333,98.500000,100.0,100.0,100.0,100.000000,33.333333,75.0,0.860,STU0187,0


In [5]:
for method in ['KNN','Cosine','Weighted']:
    if method=='KNN':
        ids=knn_ids
        table=pd.DataFrame({'student_id':candidates.iloc[ids]['student_id'].values})
    elif method=='Cosine':
        ids=cosine_ids
        table=pd.DataFrame({'student_id':candidates.iloc[ids]['student_id'].values})
    else:
        table=weighted[['student_id','overall']]
    print(method)
    display(table)

KNN


,student_id
0,STU0721
1,STU0504
2,STU1432
3,STU0038
4,STU0317


Cosine


,student_id
0,STU0721
1,STU0504
2,STU1432
3,STU0038
4,STU1553


Weighted


,student_id,overall
0,STU1710,91.925000
1,STU0093,91.875000
2,STU0317,91.591667
3,STU0076,91.433333
4,STU0187,90.675000


## Hard constraints
Room type, AC preference, food preference, and preferred environment can be applied as hard filters before ranking. They are separate from soft weighted scores.

In [6]:
weighted_recommend(query,candidates,top_k=5,constraints={'room_type':'Double','ac_preference':'Yes'})

,overall,sleep,study,cleanliness,noise,social,guest,music,preferences,sleep_difference_hours,student_id,cluster
0,91.925000,87.541667,98.333333,100.0,75.0,100.0,100.000000,66.666667,100.0,1.495,STU1710,0
1,91.875000,84.291667,97.888889,100.0,100.0,100.0,66.666667,66.666667,100.0,1.885,STU0093,0
2,91.591667,91.791667,95.444444,75.0,100.0,100.0,100.000000,66.666667,100.0,0.985,STU0317,0
3,89.950000,81.958333,97.055556,100.0,75.0,100.0,100.000000,100.000000,75.0,2.165,STU0952,1
4,89.933333,84.333333,98.777778,75.0,100.0,100.0,100.000000,100.000000,75.0,1.880,STU1373,0


In [7]:
weighted.to_csv(PROJECT_DIR/'data/dormmatch_recommendation_example.csv',index=False)
print('Saved example recommendations')

Saved example recommendations
